# 02 · Bid-ask bounce decomposition  ⭐

**Story — the credibility check.** A cross-sectional laggard signal can look far better than it is, because the fund it flags as cheapest is often cheap *because of a downward bid-ask bounce* that reverts next bar. Filling at that same close books the reversion as alpha. This notebook separates the bounce from the real edge.

> Synthetic sample; illustrative. On **real** data the same-bar upper bound was about **+16.0%/yr** and the executable estimate about **+7.9%/yr** — roughly half the apparent edge was bounce (README).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import qdii_fve

ROOT = Path(qdii_fve.__file__).resolve().parents[2]
SAMPLE = ROOT / "data" / "sample"

bars = pd.read_csv(SAMPLE / "SYNTHETIC_daily_bars.csv", parse_dates=["date"])
nav = pd.read_csv(SAMPLE / "SYNTHETIC_nav.csv", parse_dates=["date"])
und = pd.read_csv(SAMPLE / "SYNTHETIC_ndx_underlying.csv", parse_dates=["date"]).set_index("date")["close"]
fx = pd.read_csv(SAMPLE / "SYNTHETIC_usdcnh.csv", parse_dates=["date"]).set_index("date")

close = bars.pivot(index="date", columns="etf_id", values="close")
bid = bars.pivot(index="date", columns="etf_id", values="bid")
ask = bars.pivot(index="date", columns="etf_id", values="ask")
nav_pub = nav.pivot(index="date", columns="etf_id", values="nav").reindex(close.index)
print("SYNTHETIC sample — illustrative only, not the real figures.")
print(f"{close.shape[1]} funds, {close.shape[0]} days")

In [ ]:
from qdii_fve.fair_value.reconstruct import reconstruct_series

# NAV here is one session lagged, so its strike date is the previous day: scale
# the published NAV by the underlying and USDCNH move since then. (The library's
# roll_forward_nav handles arbitrary vintages; this sample uses a 1-day lag.)
u_prev = und.shift(1)
fx_prev = fx["usdcnh"].shift(1)
fair_value = pd.DataFrame(index=close.index, columns=close.columns, dtype=float)
for f in close.columns:
    fair_value[f] = reconstruct_series(
        nav_value=nav_pub[f],
        nav_underlying=u_prev,
        nav_fx=fx_prev,
        current_underlying=und,
        current_fx=fx["usdcnh"],
    )
fair_value.head()

## Build the signal, then fill it two ways

Signal = cross-sectional premium deviation (lower = cheaper), computed from the **observed** close (which carries bounce). Then:

- **same-bar close fill** — transact at the very close the signal read (bounce-inflated *upper bound*);
- **next-session opponent fill** — next day, buy the ask / sell the bid (bounce-free, *executable*).

Parameters come from the (placeholder) config, never hardcoded.

In [ ]:
from qdii_fve.backtest.signal import cross_sectional_dev, observed_premium_bps
from qdii_fve.backtest.bounce import decompose_bounce
from qdii_fve.config import is_using_example, load_config

cfg = load_config()
if is_using_example():
    print("NOTE: using placeholder config.example.yaml — parameters are NOT the deployed values.")

premium_bps = observed_premium_bps(close, fair_value)
dev = cross_sectional_dev(premium_bps, lookback=cfg.signal.norm_lookback_days)
result = decompose_bounce(dev, close, bid, ask, threshold_bps=cfg.signal.switch_threshold_bps)
result.summary()

## The two curves — the gap is bounce

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(result.same_bar_excess.index, result.same_bar_excess * 100,
        label=f"same-bar close (upper bound): {result.same_bar_ann * 100:+.1f}%/yr", lw=1.4)
ax.plot(result.executable_excess.index, result.executable_excess * 100,
        label=f"next-session opponent (executable): {result.executable_ann * 100:+.1f}%/yr", lw=1.4)
ax.fill_between(result.same_bar_excess.index, result.executable_excess * 100,
                result.same_bar_excess * 100, alpha=0.15, label="bounce component")
ax.set_title("Cumulative excess vs basket: bounce-inflated vs executable (synthetic)")
ax.set_ylabel("cumulative excess, %")
ax.legend()
plt.tight_layout()
print(f"bounce fraction of the gross edge: {result.bounce_fraction:.0%}")

## Why this is the most important artifact

The honest, executable number is the **next-session opponent-side fill** — it forgoes the bounce and pays the spread you would really cross. Reporting the same-bar figure would roughly double the apparent edge. On the real study the split was **+16.0% → +7.9%/yr**: about half of a competing signal's apparent alpha was bounce, not edge — which is why that signal is treated with caution and the executable estimate is the one that counts.

The spread charged on the executable side is the data-implied Corwin-Schultz half-spread (see [`costs`](../src/qdii_fve/costs)); a cost **ladder**, not a point estimate, then shows where even the executable edge survives.